# AADS-ULoRA Colab: Phase 3 Training (CoNeC-LoRA)

This notebook trains the Phase 3 CoNeC-LoRA (Contrastive Learning with Orthogonal Regularization) for domain-incremental learning with OOD detection.

## What this notebook does:
1. **Load Phase 2 adapter** - Load SD-LoRA augmented model
2. **Initialize CoNeC-LoRA** - Set up contrastive learning framework
3. **Train with OOD detection** - Prototype-based anomaly detection
4. **Monitor training** - Real-time metrics and memory usage
5. **Save checkpoints** - To Google Drive

---
**Expected Runtime:** 2-4 hours (depending on GPU)
**Required GPU:** Any GPU (T4, P100, A100)

---

## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path
import json
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def gate_check(step_id: str, check_name: str, condition: bool, expected: str, actual: str):
    status = "PASS" if condition else "FAIL"
    print(f"[{step_id}] {status} :: {check_name} | expected={expected} | actual={actual}")
    if not condition:
        raise RuntimeError(f"Gate failed at {step_id}::{check_name}")

# Add workspace to path
workspace_dir = Path('/content/aads_ulora')
sys.path.insert(0, str(workspace_dir))
gate_check('PHASE3_SETUP', 'workspace_exists', workspace_dir.exists(), 'existing workspace directory', str(workspace_dir))

# Load Colab configuration
config_path = workspace_dir / 'config' / 'colab.json'
gate_check('PHASE3_SETUP', 'config_exists', config_path.exists(), 'colab.json exists', str(config_path))
with open(config_path, 'r') as f:
    config = json.load(f)

gate_check('PHASE3_SETUP', 'phase3_config_present', 'phase3' in config.get('training', {}), 'training.phase3 config present', str(list(config.get('training', {}).keys())))
logger.info(f"✅ Configuration loaded from: {config_path}")
logger.info(f"GPU Type: {config['colab']['gpu_type']}")
logger.info(f"Batch size: {config['training']['phase3']['batch_size']}")

## 2. Import Dependencies

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import time
from tqdm.notebook import tqdm
import psutil
import gc

# Import training module
from src.training.colab_phase3_conec_lora import ColabPhase3Trainer, CoNeCConfig
from src.utils.model_utils import extract_pooled_output

# Import data utilities
from src.dataset.colab_datasets import ColabDomainShiftDataset
from src.dataset.colab_dataloader import ColabDataLoader

gate_check('PHASE3_IMPORTS', 'trainer_and_data_imports', True, 'all imports succeed', 'imports completed')

# Check CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

gate_check('PHASE3_IMPORTS', 'device_ready', True, 'torch initialized', str(torch.device('cuda' if torch.cuda.is_available() else 'cpu')))

## 3. Load Datasets with Domain Shift

In [ ]:
# Data paths
data_dir = workspace_dir / 'data'
metadata_path = data_dir / 'dataset_metadata.json'

# Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print("Dataset Metadata:")
print(json.dumps(metadata, indent=2))

# Create datasets with domain shift simulation
from torchvision import transforms

image_size = config['data']['image_size']
normalize_mean = config['data']['normalize_mean']
normalize_std = config['data']['normalize_std']

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

# Assuming data is organized by class folders
train_path = data_dir / 'plantvillage' / 'train'
val_path = data_dir / 'plantvillage' / 'val'

# Use domain shift dataset for Phase 3
train_dataset = ColabDomainShiftDataset(train_path, transform=train_transform, domain_label=0)
val_dataset = ColabDomainShiftDataset(val_path, transform=val_transform, domain_label=1)

print(f"\n✅ Datasets loaded:")
print(f"  Train: {len(train_dataset)} samples, {len(train_dataset.classes)} classes")
print(f"  Val: {len(val_dataset)} samples, {len(val_dataset.classes)} classes")
print(f"  Domain shift simulation enabled")

## 4. Load Phase 2 Adapter

In [ ]:
# Load Phase 2 adapter
phase2_adapter_path = workspace_dir / 'models' / 'phase2_sd_lora_adapter'

if phase2_adapter_path.exists():
    print(f"✅ Phase 2 adapter found at: {phase2_adapter_path}")
else:
    print(f"⚠️  Phase 2 adapter not found. Please run Phase 2 training first.")
    print(f"Expected path: {phase2_adapter_path}")
    raise FileNotFoundError(f"Phase 2 adapter not found")

# Get Phase 3 config
phase3_config = config['training']['phase3']

# Create CoNeC config
conec_config = CoNeCConfig(
    lora_r=phase3_config['lora_r'],
    lora_alpha=phase3_config['lora_alpha'],
    lora_dropout=phase3_config['lora_dropout'],
    learning_rate=phase3_config['learning_rate'],
    num_epochs=phase3_config['num_epochs'],
    batch_size=phase3_config['batch_size'],
    device='cuda',
    temperature=phase3_config['conec']['temperature'],
    prototype_dim=phase3_config['conec']['prototype_dim'],
    num_prototypes=phase3_config['conec']['num_prototypes'],
    contrastive_weight=phase3_config['conec']['contrastive_weight'],
    orthogonal_weight=phase3_config['conec']['orthogonal_weight'],
    gradient_accumulation_steps=config['colab']['training']['gradient_accumulation_steps'],
    use_amp=config['colab']['memory_optimization']['mixed_precision'],
    checkpoint_interval=phase3_config['checkpoint_interval'],
    early_stopping_patience=config['colab']['training']['early_stopping_patience']
)

# Create trainer
trainer = ColabPhase3Trainer(
    config=conec_config,
    checkpoint_dir=str(workspace_dir / 'checkpoints' / 'phase3')
)

# Setup optimizer
trainer.setup_optimizer()

print("✅ Trainer initialized")
print(f"  Model: {phase3_config['model_name']}")
print(f"  LoRA rank: {phase3_config['lora_r']}")
print(f"  Learning rate: {phase3_config['learning_rate']}")
print(f"  Batch size: {phase3_config['batch_size']}")
print(f"  Epochs: {phase3_config['num_epochs']}")
print(f"  CoNeC Temperature: {phase3_config['conec']['temperature']}")
print(f"  Prototypes: {phase3_config['conec']['num_prototypes']} x {phase3_config['conec']['prototype_dim']}")

## 5. Train Model

In [ ]:
# Create data loaders
batch_size = phase3_config['batch_size']

train_loader = ColabDataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=config['colab']['training']['num_workers'],
    pin_memory=config['colab']['training']['pin_memory'],
    prefetch_factor=config['colab']['training']['prefetch_factor']
)

val_loader = ColabDataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=config['colab']['training']['num_workers'],
    pin_memory=config['colab']['training']['pin_memory']
)

# Train
print("\n" + "="*60)
print("🚀 Starting Phase 3 CoNeC-LoRA Training")
print("="*60 + "\n")

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=phase3_config['num_epochs'],
    save_dir=str(workspace_dir / 'checkpoints' / 'phase3')
)

print("\n" + "="*60)
print("✅ Phase 3 Training Complete!")
print("="*60)

## 6. Plot Training Results

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Contrastive Loss
    axes[1].plot(history['contrastive_loss'], label='Contrastive Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Contrastive Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    # Orthogonal Loss
    axes[2].plot(history['orthogonal_loss'], label='Orthogonal Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss')
    axes[2].set_title('Orthogonal Loss')
    axes[2].legend()
    axes[2].grid(True)
    
    # Accuracy
    axes[3].plot(history['accuracy'], label='Accuracy')
    axes[3].set_xlabel('Epoch')
    axes[3].set_ylabel('Accuracy')
    axes[3].set_title('Validation Accuracy')
    axes[3].legend()
    axes[3].grid(True)
    
    # Learning Rate
    axes[4].plot(history['learning_rate'])
    axes[4].set_xlabel('Epoch')
    axes[4].set_ylabel('Learning Rate')
    axes[4].set_title('Learning Rate Schedule')
    axes[4].grid(True)
    
    # GPU Memory
    if 'gpu_memory' in history and history['gpu_memory']:
        memory_values = [m['allocated_gb'] for m in history['gpu_memory']]
        axes[5].plot(memory_values)
        axes[5].set_xlabel('Step')
        axes[5].set_ylabel('GPU Memory (GB)')
        axes[5].set_title('GPU Memory Usage')
        axes[5].grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot results
plot_training_history(trainer.history)

print("\nFinal metrics:")
print(f"  Best validation loss: {min(trainer.history['val_loss']):.4f}")
print(f"  Final training loss: {trainer.history['train_loss'][-1]:.4f}")
print(f"  Final validation loss: {trainer.history['val_loss'][-1]:.4f}")
print(f"  Best accuracy: {max(trainer.history['accuracy']):.4f}")

## 7. OOD Detection Evaluation

In [ ]:
# Evaluate OOD detection performance
print("\n" + "="*60)
print("🔍 Evaluating OOD Detection")
print("="*60)

# Get prototype embeddings
prototypes = trainer.prototype_manager.get_prototypes()
print(f"Prototype shape: {prototypes.shape}")

# Compute OOD metrics on validation set
ood_metrics_list = []
with torch.no_grad():
    for batch in val_loader:
        images = batch['images'].to(trainer.device)
        labels = batch['labels'].to(trainer.device)
        
        pooled = extract_pooled_output(trainer.model, images)
        ood_metrics = trainer._perform_ood_detection(pooled, labels)
        ood_metrics_list.append(ood_metrics)

# Aggregate OOD metrics
if ood_metrics_list:
    avg_prototype_dist = np.mean([m['prototype_distances'].mean().item() for m in ood_metrics_list])
    avg_mahalanobis_score = np.mean([m.get('mahalanobis_scores', torch.tensor(0)).mean().item() for m in ood_metrics_list])
    
    print(f"\nOOD Detection Metrics:")
    print(f"  Average prototype distance: {avg_prototype_dist:.4f}")
    print(f"  Average Mahalanobis score: {avg_mahalanobis_score:.4f}")
    print(f"  Prototype-based OOD rate: {np.mean([m['prototype_anomaly'].mean().item() for m in ood_metrics_list]):.2%}")
    if 'mahalanobis_anomaly' in ood_metrics_list[0]:
        print(f"  Mahalanobis OOD rate: {np.mean([m['mahalanobis_anomaly'].mean().item() for m in ood_metrics_list]):.2%}")

print("\n✅ OOD evaluation complete")

## 8. Save Final Model

In [ ]:
# Save final adapter
save_path = workspace_dir / 'models' / 'phase3_conec_lora_adapter'
trainer.save_checkpoint(str(save_path), trainer.current_epoch, trainer.best_val_loss)

print(f"✅ Model saved to: {save_path}")

# Save training history
history_path = workspace_dir / 'logs' / 'phase3_history.json'
history_path.parent.mkdir(parents=True, exist_ok=True)

with open(history_path, 'w') as f:
    json.dump(trainer.history, f, indent=2)

print(f"✅ Training history saved to: {history_path}")

# Save OOD metrics
ood_path = workspace_dir / 'logs' / 'ood_metrics.json'
with open(ood_path, 'w') as f:
    json.dump({
        'prototype_embeddings': prototypes.cpu().numpy().tolist() if prototypes is not None else None,
        'ood_metrics_history': ood_metrics_list,
        'avg_prototype_distance': float(avg_prototype_dist) if 'avg_prototype_dist' in locals() else None,
        'avg_mahalanobis_score': float(avg_mahalanobis_score) if 'avg_mahalanobis_score' in locals() else None
    }, f, indent=2)

print(f"✅ OOD metrics saved to: {ood_path}")

print("\n🎉 Phase 3 complete! All training phases finished.")